# Holiday Identification by Daily Profile Clustering

This notebook flags atypical days in one demand series by comparing each daily profile against its `(segment, weekday)` reference group.

It loads observed data, builds day-level profiles, measures distance to the group centroid, and compares detected outliers against the local holiday catalog.

This is exploratory notebook code, not part of the production pipeline.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import analog_holidays.shared.identify_holidays as identify_holidays_module
identify_holidays_module = importlib.reload(identify_holidays_module)

from analog_holidays.shared.dataset_config import ACTIVE_CONFIG, list_dataset_regions
from analog_holidays.shared.identify_holidays import (
    build_holiday_groups,
    build_holiday_selector_features,
    build_wide_df,
    compare_outliers_holidays,
    compute_distances,
    detect_outliers,
    display_cluster_holiday_crosstab,
    display_cluster_38h_crosstab,
    find_holidays_not_detected,
    get_date_sets,
    get_hour_cols,
    load_holidays_catalog,
    load_results_data,
    plot_distance_distribution,
    plot_profiles_by_holiday,
    plot_profiles_by_segment_dow,
    print_summary,
    report_nth_monday_holidays,
    run_analog_cluster_38h_analysis,
    run_cluster_ab_validation,
    run_cluster_38h_analysis,
    run_cluster_atypical_analysis,
    run_holiday_ab_validation,
 )

DEMAND_PATH = ACTIVE_CONFIG.demand_path
HOLIDAYS_PATH = ACTIVE_CONFIG.notebook_holidays_path

print(f'Active dataset: {ACTIVE_CONFIG.key}')
print(f'Demand CSV: {DEMAND_PATH}')
print(f'Holidays: {HOLIDAYS_PATH}')


Active dataset: ercot
Demand CSV: /home/uriel/GIT/analog_holidays/holidays/holiday_demand_ercot.csv
Holidays: /home/uriel/GIT/analog_holidays/docs/ercot_holidays.json


## 1. Configuration

In [2]:
UNIQUE_ID = 'ERCOT_demand_ERCOT'

DATE_END = None
SELECTOR_FUTURE_END = '2026-12-31'
CLUSTERING_CRITERIUM = 'best_matching_weekday'
CLUSTERING_CRITERIA_CATALOG = identify_holidays_module.ANALOG_CLUSTER_CRITERIA_CATALOG.copy()
# Public criterion catalog exposed by the selector pipeline:
# - 'shape_pearson_CDE_map_FGH' keeps the current 38h shape-based C/D/E -> F/G/H mapping.
# - 'seasonal_heat_cold' groups holiday dates into heat vs cold seasons, mapping Spring/Summer to heat and Autumn/Winter to cold.
# - 'seasonal_winter_sprint_fall' groups holiday dates by year season and maps Winter/Spring/Fall/Summer to stable analog labels.
# - 'best_matching_weekday' groups holiday dates by the closest weekday-profile label assigned to each date.
# - 'observance_tier' groups holiday dates by how inhábil they actually are (working/partial/full), derived from observed_strength.


AVAILABLE_UNIQUE_IDS = list_dataset_regions()
if not AVAILABLE_UNIQUE_IDS:
    raise ValueError(f'No configured series were found in {DEMAND_PATH}.')
if UNIQUE_ID is None:
    UNIQUE_ID = AVAILABLE_UNIQUE_IDS[0]
elif UNIQUE_ID not in AVAILABLE_UNIQUE_IDS:
    raise ValueError(
        f'Series {UNIQUE_ID!r} not found. Available: {AVAILABLE_UNIQUE_IDS}'
    )

MONTH_NAMES = [
    'January', 'February', 'March', 'April', 'May', 'June',
    'July', 'August', 'September', 'October', 'November', 'December',
]

SEGMENTS = [
    {'label': 'Spring', 'months': [3, 4, 5]},
    {'label': 'Summer', 'months': [6, 7, 8]},
    {'label': 'Autumn', 'months': [9, 10, 11]},
    {'label': 'Winter', 'months': [12, 1, 2]},
]

SEGMENTS = [{'label': month_name, 'months': [month_number]} for month_number, month_name in enumerate(MONTH_NAMES, start=1)]

OUTLIER_PERCENTILE = 95
EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE = True
KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE = 75
DISTANCE_METRIC = 'PEARSON'

N_CLUSTERS_PHOL    = 3   # clusters for 38-h eve+holiday profiles (section 8c-bis)
PREVIOUSLY_W_HOURS = 14  # hours taken from the eve day

WEEKDAY_NAMES = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

print(f'Series:   {UNIQUE_ID}')
print(f'DATE_END: {DATE_END}')
print(f'SELECTOR_FUTURE_END: {SELECTOR_FUTURE_END}')
print(f'CLUSTERING_CRITERIUM: {CLUSTERING_CRITERIUM}')
print(f'Criterion catalog: {tuple(CLUSTERING_CRITERIA_CATALOG)}')
print(f'Available series: {AVAILABLE_UNIQUE_IDS}')
display(
    pd.DataFrame(
        [
            {'criterion': criterion_name, 'description': criterion_description}
            for criterion_name, criterion_description in CLUSTERING_CRITERIA_CATALOG.items()
        ]
    )
)


Series:   ERCOT_demand_ERCOT
DATE_END: None
SELECTOR_FUTURE_END: 2026-12-31
CLUSTERING_CRITERIUM: best_matching_weekday
Criterion catalog: ('shape_pearson_CDE_map_FGH', 'seasonal_heat_cold', 'seasonal_winter_sprint_fall', 'best_matching_weekday', 'observance_tier', 'observance_tier_depth')
Available series: ['ERCOT_demand_COAST', 'ERCOT_demand_EAST', 'ERCOT_demand_FWEST', 'ERCOT_demand_NORTH', 'ERCOT_demand_NCENT', 'ERCOT_demand_SOUTH', 'ERCOT_demand_SCENT', 'ERCOT_demand_WEST', 'ERCOT_demand_ERCOT']


,criterion,description
0,shape_pearson_CDE_map_FGH,Maps the current 38h shape-based event_profile...
1,seasonal_heat_cold,Groups holiday dates into heat vs cold seasons...
2,seasonal_winter_sprint_fall,Groups holiday dates by year season and maps W...
3,best_matching_weekday,"Groups holiday dates by best_matching_weekday,..."
4,observance_tier,Groups holiday dates by how inhábil they actua...
5,observance_tier_depth,Four-tier refinement of observance_tier that s...


## 2. Load observed data from CSV

In [3]:
MONTH_TO_SEGMENT = {month: segment['label'] for segment in SEGMENTS for month in segment['months']}
SEGMENT_LABELS = [segment['label'] for segment in SEGMENTS]

df_raw = load_results_data(DEMAND_PATH, UNIQUE_ID)
if DATE_END is not None:
    cutoff_ts = pd.Timestamp(DATE_END)
    df_raw = df_raw[df_raw['ds'] < cutoff_ts].copy()
    if df_raw.empty:
        raise ValueError(f'No data available for {UNIQUE_ID} before {cutoff_ts.date()}.')

print(f'Series: {UNIQUE_ID}')
print(f'Records: {len(df_raw):,}')
if DATE_END is not None:
    print(f'DATE_END cutoff: {pd.Timestamp(DATE_END).date()} (exclusive)')
print(f'Range: {df_raw["ds"].min()} → {df_raw["ds"].max()}')
df_raw.head()

Series: ERCOT_demand_ERCOT
Records: 81,061
Range: 2016-01-01 01:00:00 → 2026-04-01 00:00:00


,ds,y
0,2016-01-01 01:00:00,33852.758587
1,2016-01-01 02:00:00,33434.097940
2,2016-01-01 03:00:00,33099.066023
3,2016-01-01 04:00:00,33053.807611
4,2016-01-01 05:00:00,33441.899067


## 3. Pivot to wide format (1 row = 1 day, 24 columns = hours)

In [4]:
df_wide = build_wide_df(df_raw, MONTH_TO_SEGMENT, exclude_years=[])
HOUR_COLS = get_hour_cols(df_wide)
year_min = int(df_wide.index.year.min())
year_max = int(df_wide.index.year.max())
df_holidays = load_holidays_catalog(HOLIDAYS_PATH, year_min, year_max)
df_holidays_display = df_holidays.copy()
KNOWN_HOLIDAY_DATES = (
    set(pd.to_datetime(df_holidays['date']).dt.normalize())
    if EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE else set()
 )

print(f'Complete days: {len(df_wide)}')
print(f'Segments ({len(SEGMENT_LABELS)}):')
for segment_label in SEGMENT_LABELS:
    n_days = (df_wide['segment'] == segment_label).sum()
    print(f'  {segment_label}: {n_days} days')
print(f'Catalog holidays in range: {len(df_holidays)}')
print(f'Dates excluded from the baseline: {len(KNOWN_HOLIDAY_DATES)}')
df_wide.head()

Complete days: 3366
Segments (12):
  January: 309 days
  February: 283 days
  March: 300 days
  April: 270 days
  May: 279 days
  June: 270 days
  July: 279 days
  August: 279 days
  September: 270 days
  October: 279 days
  November: 269 days
  December: 279 days
Catalog holidays in range: 165
Dates excluded from the baseline: 165


hour,0,1,2,3,4,5,6,7,8,9,...,17,18,19,20,21,22,23,dow,month,segment
date,,,,,,,,,,,,,,,,,,,,,
2016-01-02,37005.22770,35639.572798,34751.639519,34190.896238,33907.215963,34180.104903,35017.067408,36244.127179,37557.222730,38738.557297,...,38243.331969,40018.295132,41322.819874,41060.155780,40569.459516,39659.740810,38214.586273,5,1,January
2016-01-03,36511.74022,35150.461293,34305.605861,33940.879126,33960.450483,34377.966501,35319.341299,36688.311653,37985.559483,38541.844568,...,32134.343790,34112.210254,37128.108786,37908.137843,37987.628461,37607.971249,36278.512948,6,1,January
2016-01-04,34745.45126,33858.741394,33686.620802,34027.365303,34856.339419,36609.483990,39915.596086,44061.239800,45766.623422,44706.793377,...,34968.061604,37134.083400,40482.388764,41218.815306,41443.449614,40608.985437,38585.122231,0,1,January
2016-01-05,36625.83313,35515.542099,35232.654282,35377.808446,36008.786557,37671.019569,41036.532006,45621.767043,46975.142846,45787.747186,...,37898.317552,40373.364188,42856.610632,42911.586032,42428.270434,40906.578073,38343.718856,1,1,January
2016-01-06,35797.85395,34194.323311,33413.596310,33050.056849,33073.061890,34042.335661,36670.616218,40659.974359,41849.702712,40989.723486,...,38904.616583,40572.200388,41744.554058,41284.160433,40396.203014,38603.184940,35900.716213,2,1,January


## 4. Compute distances by segment and weekday

Distances are computed inside each `(segment, dow)` group against that group's reference centroid.

`DISTANCE_METRIC` controls whether the score emphasizes magnitude, shape, or both.

In [5]:
df_dist = compute_distances(
    df_wide,
    HOUR_COLS,
    SEGMENT_LABELS,
    WEEKDAY_NAMES,
    distance_metric=DISTANCE_METRIC,
    reference_exclude_dates=KNOWN_HOLIDAY_DATES,
 )

print(f'Distance metric: {DISTANCE_METRIC}')
if DISTANCE_METRIC == 'PEARSON_EUCLIDIAN':
    print('  Euclidean and Pearson components combined after group-wise normalization')
elif DISTANCE_METRIC == 'EUCLIDIAN':
    print('  Euclidean distance only')
elif DISTANCE_METRIC == 'PEARSON':
    print('  1 - Pearson r only')
print(f'\nSegments: {SEGMENT_LABELS}')
print(f'Total records (day × group): {len(df_dist)}')
df_dist.head(10)

Distance metric: PEARSON
  1 - Pearson r only

Segments: ['January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September', 'October', 'November', 'December']
Total records (day × group): 3366


,date,segment,dow,dow_name,dist_eucl,dist_pearson,dist_eucl_norm,dist_pearson_norm,is_reference_day,distance
0,2016-01-04,January,0,Monday,26135.420870,0.086216,-0.373788,-0.441939,True,-0.441939
1,2016-01-11,January,0,Monday,12116.869678,0.137521,-0.970111,-0.168274,True,-0.168274
2,2016-01-18,January,0,Monday,21380.968678,0.124017,-0.576034,-0.240303,False,-0.240303
3,2016-01-25,January,0,Monday,50675.210016,0.072892,0.670089,-0.513009,True,-0.513009
4,2017-01-02,January,0,Monday,57726.422286,0.455961,0.970035,1.530312,True,1.530312
5,2017-01-09,January,0,Monday,27561.768450,0.460629,-0.313114,1.555210,True,1.555210
6,2017-01-16,January,0,Monday,47005.311629,0.266552,0.513979,0.519990,False,0.519990
7,2017-01-23,January,0,Monday,46208.453119,0.014273,0.480082,-0.825686,True,-0.825686
8,2017-01-30,January,0,Monday,41873.988542,0.050570,0.295702,-0.632073,True,-0.632073
9,2018-01-01,January,0,Monday,49470.781910,0.144545,0.618855,-0.130808,False,-0.130808


## 6. Load the local recognized holidays catalog (`holidays/holidays_recognized.json`)

In [6]:
df_dist, df_outliers = detect_outliers(
    df_dist,
    OUTLIER_PERCENTILE,
    threshold_reference_only=EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE,
    promote_dates=KNOWN_HOLIDAY_DATES,
    promote_min_group_percentile=KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE,
)

print(f'Detected outliers: {len(df_outliers)}')
print(f'Out of a total of {len(df_dist)} days')
print(f'Percentage: {100 * len(df_outliers) / len(df_dist):.1f}%')
print(f'Known holidays rescued by within-group percentile: {int(df_dist["is_promoted_outlier"].sum())}')
df_outliers.head(20)

Detected outliers: 245
Out of a total of 3366 days
Percentage: 7.3%
Known holidays rescued by within-group percentile: 66


,date,segment,dow,dow_name,dist_eucl,dist_pearson,dist_eucl_norm,dist_pearson_norm,is_reference_day,distance,threshold,group_percentile,is_outlier,is_promoted_outlier
1201,2025-05-26,May,0,Monday,58351.141236,0.262541,1.604502,10.844884,False,10.844884,1.001917,100.0,True,True
1800,2018-07-04,July,2,Wednesday,56370.273162,0.060063,2.119716,7.999144,False,7.999144,1.161631,100.0,True,True
2787,2017-10-29,October,6,Sunday,55318.701459,0.868005,1.302647,6.119478,True,6.119478,0.409836,100.0,True,False
1640,2017-06-24,June,5,Saturday,53192.290087,0.198594,1.160111,6.101876,True,6.101876,0.216977,100.0,True,False
1744,2024-07-08,July,0,Monday,52211.253507,0.281284,1.783952,6.085908,True,6.085908,0.334432,100.0,True,False
943,2019-04-02,April,1,Tuesday,36026.603796,0.893035,0.743641,5.873242,True,5.873242,0.846251,100.0,True,False
2611,2023-10-31,October,1,Tuesday,30859.740854,0.937666,-0.119485,5.845030,True,5.845030,0.713707,100.0,True,False
1316,2024-05-16,May,3,Thursday,27757.577735,0.348821,-0.122055,5.799566,True,5.799566,0.417747,100.0,True,False
2473,2018-09-22,September,5,Saturday,52214.516749,0.114143,1.153436,5.573552,True,5.573552,1.182491,100.0,True,False
836,2022-03-12,March,5,Saturday,57674.047482,1.828848,2.060218,5.536366,True,5.536366,0.748567,100.0,True,False


## 6. Load the local recognized holidays catalog (`holidays/holidays_recognized.json`)

In [7]:
print(f'Generated holidays: {len(df_holidays_display)} (from {year_min} to {year_max})')
df_holidays_display.tail(15)


Generated holidays: 165 (from 2016 to 2026)


,date,holiday_name
150,2026-01-01,New Year's Day
151,2026-01-19,Martin Luther King Jr. Day
152,2026-02-16,Presidents' Day
153,2026-03-02,Texas Independence Day
154,2026-04-21,San Jacinto Day
155,2026-05-25,Memorial Day
156,2026-06-19,Juneteenth National Independence Day
157,2026-07-04,Independence Day
158,2026-08-27,Lyndon B. Johnson Day
159,2026-09-07,Labor Day


## 6b. Holidays with an nth-Monday rule

Three civic holidays moved from fixed dates to observed Mondays after the 2006 Federal Labor Law reform. Years before 2006 keep the historical fixed date.

| Holiday | Before 2006 | Since 2006 |
|---|---|---|
| Constitution Day | February 5 | 1st Monday of February |
| Benito Juarez's Birthday | March 21 | 3rd Monday of March |
| Mexican Revolution Day | November 20 | 3rd Monday of November |

In [8]:
df_nth = report_nth_monday_holidays(df_holidays)

print(f'Holidays with an nth-Monday rule (since 2006): {len(df_nth)} occurrences')
display(
    df_nth.rename(columns={
        'holiday_name': 'Holiday',
        'date': 'Observed date',
        'weekday_name': 'Weekday',
        'labor_law_rule': 'LFT rule',
    })
)

Holidays with an nth-Monday rule (since 2006): 0 occurrences


,Holiday,Observed date,Weekday,LFT rule


## 7. Compare outliers vs known holidays

In [9]:
df_match, stats = compare_outliers_holidays(df_outliers, df_holidays)
df_outliers_cmp = df_match
df_match_display = df_match.copy()

n_total = stats['n_total']
n_match = stats['n_match']
n_unknown = stats['n_unknown']

print('Outlier comparison against the holiday catalog')
print(f'Total outliers: {n_total}')
print(f'Match a known holiday: {n_match} ({100 * n_match / n_total:.0f}%)')
print(f'No catalog match: {n_unknown} ({100 * n_unknown / n_total:.0f}%)')

cols_display = ['date', 'dow_name', 'segment', 'distance', 'holiday_name']

print('\nKnown holidays detected as outliers')
display(
    df_match_display[df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

print('\nOutliers without a catalog match')
display(
    df_match_display[~df_match_display['is_known_holiday']]
    .sort_values('holiday_name')[cols_display]
 )

Outlier comparison against the holiday catalog
Total outliers: 245
Match a known holiday: 66 (27%)
No catalog match: 179 (73%)

Known holidays detected as outliers


,date,dow_name,segment,distance,holiday_name
178,2017-12-25,Monday,December,1.772871,Christmas Day
191,2025-12-25,Thursday,December,1.371775,Christmas Day
214,2024-12-25,Wednesday,December,0.688778,Christmas Day
34,2020-12-25,Friday,December,4.410528,Christmas Day
208,2018-12-25,Tuesday,December,0.800222,Christmas Day
...,...,...,...,...,...
238,2022-11-24,Thursday,November,0.088236,Thanksgiving Day
139,2018-11-22,Thursday,November,2.393929,Thanksgiving Day
134,2017-11-23,Thursday,November,2.502443,Thanksgiving Day
201,2025-11-11,Tuesday,November,1.012201,Veterans Day



Outliers without a catalog match


,date,dow_name,segment,distance,holiday_name
2,2017-10-29,Sunday,October,6.119478,NaN
3,2017-06-24,Saturday,June,6.101876,NaN
4,2024-07-08,Monday,July,6.085908,NaN
5,2019-04-02,Tuesday,April,5.873242,NaN
6,2023-10-31,Tuesday,October,5.845030,NaN
...,...,...,...,...,...
205,2025-04-08,Tuesday,April,0.853533,NaN
209,2020-03-07,Saturday,March,0.750814,NaN
215,2017-10-22,Sunday,October,0.572741,NaN
222,2016-07-25,Monday,July,0.519535,NaN


In [10]:
detected_dates = set(pd.to_datetime(df_outliers_cmp['date']).dt.normalize())
all_dates_in_data = set(df_wide.index.normalize())

df_missed = find_holidays_not_detected(
    df_holidays,
    all_dates_in_data,
    detected_dates,
    WEEKDAY_NAMES,
 )
df_missed_display = df_missed.copy()

print(f'Known holidays present in the data but not detected as outliers: {len(df_missed)}')
if not df_missed.empty:
    display(df_missed_display.sort_values('holiday_name')[['holiday_name', 'dow_name', 'date']])

date_sets = get_date_sets(df_outliers_cmp, df_holidays, all_dates_in_data)
outlier_dates_set = date_sets['outlier_dates_set']
holiday_dates_set = date_sets['holiday_dates_set']
match_dates_set = date_sets['match_dates_set']
unknown_dates_set = date_sets['unknown_dates_set']
missed_dates_set = date_sets['missed_dates_set']

Known holidays present in the data but not detected as outliers: 72


,holiday_name,dow_name,date
54,Christmas Day,Monday,2023-12-25
11,Christmas Day,Sunday,2016-12-25
33,Christmas Day,Wednesday,2019-12-25
53,Christmas Eve,Sunday,2023-12-24
10,Christmas Eve,Saturday,2016-12-24
...,...,...,...
52,Veterans Day,Saturday,2023-11-11
7,Veterans Day,Friday,2016-11-11
39,Veterans Day,Wednesday,2020-11-11
46,Veterans Day,Friday,2022-11-11


## 8. Visualization — profiles by weekday and outliers

In [11]:
plot_profiles_by_segment_dow(
    df_wide, HOUR_COLS, SEGMENT_LABELS, WEEKDAY_NAMES,
    match_dates_set, unknown_dates_set, missed_dates_set, UNIQUE_ID,
    df_holidays=df_holidays_display,
)

/home/uriel/GIT/analog_holidays/shared/identify_holidays.py:1689: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


Detected profiles use the same colors as the plotting helpers:

| Color | Meaning |
|---|---|
| Green | Detected outlier that matches a catalog holiday |
| Red | Detected outlier without a catalog match |
| Blue | Catalog holiday that was not flagged as an outlier |
| Black dashed line | Group centroid |
| Faint gray | Regular days |

## 8b. Hourly profiles by holiday

Each subplot overlays all available yearly profiles for one holiday present in the data.

| Color | Meaning |
|---|---|
| Green | Year detected as an outlier |
| Blue | Year present but not detected as an outlier |
| Black dashed line | Average holiday profile across years |

In [12]:
holiday_groups = build_holiday_groups(df_holidays_display, df_wide.index)

plot_profiles_by_holiday(
    df_wide, holiday_groups, df_holidays_display, HOUR_COLS,
    match_dates_set, outlier_dates_set, UNIQUE_ID,
)

/home/uriel/GIT/analog_holidays/shared/identify_holidays.py:1799: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 8c. Cluster atypical profiles

This view clusters the detected atypical profiles and compares each cluster centroid against weekday reference profiles.

In [13]:
N_CLUSTERS = 2
CLUSTER_COLORS = [
    '#f58231',  '#4363d8', '#3cb44b', '#e6194b',
    '#911eb4', '#42d4f4', '#f032e6', '#bfef45',
]

cluster_results = run_cluster_atypical_analysis(
    df_wide,
    match_dates_set,
    set(),
    outlier_dates_set,
    N_CLUSTERS,
    HOUR_COLS,
    df_holidays_display,
    CLUSTER_COLORS,
    UNIQUE_ID,
)
df_atyp = cluster_results['df_atyp']
df_sim = cluster_results['df_sim']


Similarity between each cluster centroid and the weekday reference profiles


,n days,r vs Mon,r vs Tue,r vs Wed,r vs Thu,r vs Fri,r vs Sat,r vs Sun,top holidays
Cluster,,,,,,,,,
0,47,0.971,0.975,0.975,0.977,0.977,0.999,0.995,"Memorial Day, Independence Day, New Year's Day"
1,19,-0.317,-0.331,-0.333,-0.342,-0.340,-0.477,-0.518,"Christmas Eve, Thanksgiving Day, Christmas Day"



Days per cluster


cluster,date,dow,type,holiday_name
0,2016-05-30,Mon,Confirmed holiday,Memorial Day
0,2016-07-04,Mon,Confirmed holiday,Independence Day
0,2017-01-01,Sun,Confirmed holiday,New Year's Day
0,2017-01-16,Mon,Confirmed holiday,Martin Luther King Jr. Day
0,2017-05-29,Mon,Confirmed holiday,Memorial Day
0,2017-07-04,Tue,Confirmed holiday,Independence Day
0,2017-08-27,Sun,Confirmed holiday,Lyndon B. Johnson Day
0,2018-02-19,Mon,Confirmed holiday,Presidents' Day
0,2018-03-02,Fri,Confirmed holiday,Texas Independence Day
0,2018-05-28,Mon,Confirmed holiday,Memorial Day


/home/uriel/GIT/analog_holidays/shared/identify_holidays.py:1904: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 8c-bis. Cluster: 38-h event profile (eve last 14 h + holiday 24 h)

For every confirmed holiday `D`, a single **38-hour feature vector** is built:

| Segment | Hours | Source |
|---------|-------|--------|
| Pre-holiday eve | last `PREVIOUSLY_W_HOURS = 14` h of day `D−1` (h10…h23) | `df_wide[D−1]` |
| Holiday | full 24 h of day `D` (h0…h23) | `df_wide[D]` |

KMeans is then applied to these 38-column profiles.  
Events where the eve day is missing from the data are skipped.


In [14]:
cluster_38h_results = run_cluster_38h_analysis(
    df_wide=df_wide,
    match_dates_set=match_dates_set,
    df_holidays_display=df_holidays_display,
    hour_cols=HOUR_COLS,
    cluster_colors=CLUSTER_COLORS,
    unique_id=UNIQUE_ID,
    n_clusters=N_CLUSTERS_PHOL,
    previously_w_hours=PREVIOUSLY_W_HOURS,
)
df_phol      = cluster_38h_results['df_phol']
df_days_phol = cluster_38h_results['df_days_phol']

Events with complete 38-h profile: 65

Days per cluster


type,date,dow,holiday_name
C,2019-08-27,Tue,Lyndon B. Johnson Day
C,2022-05-30,Mon,Memorial Day
C,2022-07-04,Mon,Independence Day
C,2022-12-24,Sat,Christmas Eve
C,2023-07-04,Tue,Independence Day
C,2023-08-27,Sun,Lyndon B. Johnson Day
C,2023-09-04,Mon,Labor Day
C,2024-05-27,Mon,Memorial Day
C,2024-06-19,Wed,Juneteenth National Independence Day
C,2024-07-04,Thu,Independence Day


Holiday × Cluster  (38-h profiles)
Rows = holiday name   |   Columns = cluster type (count of events)



/home/uriel/GIT/analog_holidays/shared/identify_holidays.py:2184: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


cluster_type,C,D,E
holiday_name,,,
Christmas Day,—,3,3
Christmas Eve,1,2,3
Day after Thanksgiving,—,1,1
Independence Day,4,3,—
Juneteenth National Independence Day,1,2,—
Labor Day,2,3,—
Lyndon B. Johnson Day,2,—,1
Martin Luther King Jr. Day,—,1,3
Memorial Day,3,2,3


## 8d. Cluster DOW similarity

For each cluster, ranks all **7 day-of-week** reference profiles (Mon–Sun) by mean Pearson correlation, then tests statistically whether the top-ranked DOW is significantly better than the runner-up.

This avoids the ad-hoc Sat/Sun assumption: a cluster that resembles Wednesday and Sunday equally will show up as "not distinguishable", while a cluster that truly behaves like Saturday will have a clear green winner column.

| Output | Meaning |
|---|---|
| **Table 1 — mean corr** | Mean Pearson r of each cluster against each DOW centroid (same month). Green = highest, pink = lowest per row. |
| **Table 2 — votes** | Number of days in the cluster where each DOW was the best individual match. |
| **Table 3 — per-day** | Every atypical day with its 7 correlation values; green = best DOW for that day. |
| `best_dow` | DOW with highest mean r (winner). |
| `runner_up_dow` | Second-best DOW. |
| `gap_corr` | Mean r of winner minus mean r of runner-up. |
| `wilcoxon_pvalue` | One-sided Wilcoxon on (r_winner − r_runner_up) per day: p < alpha → winner is unambiguously better. |
| `cluster_type` | Winner DOW label if significant, `'unclear'` otherwise. |


In [15]:
AB_ALPHA = 0.10

cluster_ab_results = run_cluster_ab_validation(
    df_wide=df_wide,
    df_atyp=df_atyp,
    outlier_dates_set=outlier_dates_set,
    hour_cols=HOUR_COLS,
    cluster_colors=CLUSTER_COLORS,
    alpha=AB_ALPHA,
    df_holidays=df_holidays_display,
)

# cluster_type: winner DOW label (e.g. 'Sat', 'Sun') if statistically clear, else 'unclear'
cluster_type_map = dict(
    zip(
        cluster_ab_results['summary_df']['cluster'],
        cluster_ab_results['summary_df']['cluster_type'],
    )
)
print('cluster_type_map =', cluster_type_map)


Cluster DOW similarity  (alpha = 0.1)
Mean Pearson r of each cluster against the 7 DOW reference profiles.



cluster,n_days,Mon,Tue,Wed,Thu,Fri,Sat,Sun,best_dow,runner_up_dow,gap_corr,wilcoxon_pvalue,decision
0,47,0.847,0.856,0.851,0.863,0.847,0.891,0.882,Sat,Sun,0.008,0.0512,"Sat-like (Δr=0.008, p=0.0512)"
1,19,0.334,0.341,0.339,0.288,0.361,0.173,0.091,Fri,Tue,0.021,0.5702,"Fri / Tue not distinguishable (Δr=0.021, p=0.5702)"



Vote counts — best-matching DOW per day


cluster,n_days,Mon,Tue,Wed,Thu,Fri,Sat,Sun,best_dow
0,47,2,3,0,0,5,19,18,Sat
1,19,0,7,5,0,7,0,0,Fri



Per-day detail


cluster,date,profile_type,best_dow,corr_Mon,corr_Tue,corr_Wed,corr_Thu,corr_Fri,corr_Sat,corr_Sun
0,2016-05-30,Memorial Day,Sun,0.964,0.964,0.971,0.964,0.957,0.984,0.993
0,2016-07-04,Independence Day,Sun,0.979,0.982,0.982,0.984,0.985,0.994,0.997
0,2017-01-01,New Year's Day,Sun,0.422,0.543,0.468,0.483,0.376,0.578,0.624
0,2017-01-16,Martin Luther King Jr. Day,Tue,0.716,0.810,0.756,0.756,0.705,0.780,0.751
0,2017-05-29,Memorial Day,Sun,0.968,0.968,0.974,0.968,0.962,0.987,0.996
0,2017-07-04,Independence Day,Sun,0.983,0.986,0.986,0.988,0.989,0.997,0.998
0,2017-08-27,Lyndon B. Johnson Day,Sun,0.966,0.968,0.965,0.968,0.962,0.976,0.983
0,2018-02-19,Presidents' Day,Tue,0.911,0.955,0.936,0.872,0.729,0.874,0.876
0,2018-03-02,Texas Independence Day,Fri,0.847,0.852,0.850,0.841,0.856,0.679,0.582
0,2018-05-28,Memorial Day,Sat,0.977,0.981,0.983,0.983,0.983,0.998,0.994


cluster_type_map = {0: 'Sat', 1: 'unclear'}


In [16]:
df_cluster_holiday_crosstab = display_cluster_holiday_crosstab(
    df_atyp=df_atyp,
    df_holidays=df_holidays_display,
    cluster_colors=CLUSTER_COLORS,
)


Holiday × Cluster distribution
Rows = holiday name   |   Columns = cluster (count of days)



cluster,Cluster 0,Cluster 1
holiday_name,,
Christmas Day,3,3
Christmas Eve,2,4
Day after Thanksgiving,—,2
Independence Day,7,—
Juneteenth National Independence Day,3,—
Labor Day,5,—
Lyndon B. Johnson Day,3,—
Martin Luther King Jr. Day,2,2
Memorial Day,8,—


## 9. Holiday Selector Feature Schema

The selector table `df_holiday_selector_features` combines calendar rules and profile-based labels so analog candidates can be filtered or ranked by context before distance ranking.

| Field | Meaning | Typical values / source |
|---|---|---|
| `unique_id` | Demand series identifier that produced the profile labels. This keeps the exported selector series-aware when multiple regions share the same CSV. | `SEN_demand_SIN`, `OCC_demand_BAJ` |
| `holiday_name` | Name assigned to the row itself. For `H1`/`H2`/`H3` rows this is the holiday on that date. For `H4` rows this is the synthetic recovery label. | `Labor Day`, `Good Friday`, `Post-holiday recovery` |
| `anchor_holiday_name` | Anchor holiday used to identify the event family behind the row. For `H1`/`H2`/`H3` it matches `holiday_name`. For `H4` it is the last holiday in the immediately preceding run. | `Christmas Day`, `New Year's Day` |
| `date` | Calendar date represented by the row. | `2020-05-01` |
| `holiday_day_type` | H-day taxonomy label. `H1` = eve day, `H2` = core or standalone holiday, `H3` = consecutive post-holiday day, `H4` = first recovery day after a run of length >= 2. | `H1`, `H2`, `H3`, `H4` |
| `weekday_name` | Literal weekday of `date`, independent of demand similarity. | `Monday`, `Saturday`, `Sunday` |
| `day_class_code` | Compact labor-calendar code used by the selector. `1` = weekday, `2` = Saturday, `3` = Sunday. | `1`, `2`, `3` |
| `day_class_name` | Expanded text version of `day_class_code`. | `Weekday`, `Saturday`, `Sunday` |
| `season` | Meteorological season derived from the month. | `Winter`, `Spring`, `Summer`, `Autumn` |
| `date_rule` | Calendar rule behind the date. `fixed_date` = fixed official date, `observed_monday_rule` = Monday-observed civic holiday after the 2006 labor reform, `movable_date` = Easter-based movable holiday, `derived_recovery_day` = synthetic `H4` row. | `fixed_date`, `observed_monday_rule`, `movable_date`, `derived_recovery_day` |
| `is_fixed_date` | Boolean helper flag for fixed-date observances. This is also `True` for pre-2006 years of Monday-observed civic holidays, when they still fell on their original fixed date. | `True`, `False` |
| `is_observed_monday_rule` | Boolean helper flag for civic holidays shifted to Monday by the 2006 labor reform. | `True`, `False` |
| `best_matching_weekday` | Best weekday match at the individual-day level, taken from section 8d (`best_dow`) and expanded to full weekday names. This is the closest weekday profile for that specific date. | `Saturday`, `Sunday`, `Wednesday` |
| `daily_profile_cluster` | DOW-agnostic letter for the daily-profile cluster from section 8c. It is derived from `daily_profile_cluster_id` in ascending numeric order, so the current notebook typically shows `A`, `B`, etc. | `A`, `B`, `C` |
| `daily_profile_cluster_id` | Raw numeric KMeans cluster id from section 8c (atypical daily profiles). | `0`, `1`, `2` |
| `daily_profile_archetype` | Cluster-level weekday archetype inferred from section 8d (`cluster_type`). This is the human-readable interpretation of the cluster, such as `Saturday-like`, `Sunday-like`, or `unclear`. | `Saturday-like`, `Sunday-like`, `unclear` |
| `event_profile_cluster` | DOW-agnostic letter for the 38-h event-profile cluster from section 8c-bis. The current notebook typically shows `C`, `D`, `E`, etc. | `C`, `D`, `E` |
| `event_profile_cluster_id` | Raw numeric KMeans cluster id from section 8c-bis (38-h eve + holiday profile). | `0`, `1`, `2` |
| `analog_cluster_criterion` | Public criterion selected in section 11 to derive `analog_cluster` in the exported selector. This records which grouping rule produced the stable analog labels. | `seasonal_heat_cold`, `best_matching_weekday` |

Notes:

- `best_matching_weekday` is a per-date label, while `daily_profile_archetype` is a cluster-level label.
- `daily_profile_cluster` / `daily_profile_cluster_id` come from the 24-h atypical-day analysis in section 8c.
- `event_profile_cluster` / `event_profile_cluster_id` come from the 38-h eve+holiday analysis in section 8c-bis.
- `analog_cluster_criterion` is constant within one selector export and tells downstream notebooks how `analog_cluster` was derived.
- `unique_id` is exported so the selector and priors can coexist for multiple series in the same CSV without mixing labels across regions.
- `H4` rows are derived automatically and may not have event-profile labels because the 38-h clustering is defined only for confirmed holiday dates.
- Missing values in cluster-related columns are acceptable when a row was not part of the corresponding upstream analysis.
- For future candidates, profile-based labels are inferred first within the same `anchor_holiday_name` + `holiday_day_type` family and, if that subtype has no observed evidence, they fall back to the broader `anchor_holiday_name` history.

In [17]:
# 9. Holiday selector feature table (historical window for all series)
import numpy as np
from analog_holidays.shared.identify_holidays import build_holiday_selector_features

_SELECTOR_GROUP_COLS = ('unique_id', 'anchor_holiday_name', 'holiday_day_type')
_SELECTOR_DOW_LABELS = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
selector_n_clusters = int(N_CLUSTERS) if 'N_CLUSTERS' in globals() else 2
selector_ab_alpha = float(AB_ALPHA) if 'AB_ALPHA' in globals() else 0.10


def _empty_cluster_ab_results():
    return {
        'occurrences_df': pd.DataFrame(),
        'summary_df': pd.DataFrame(),
        'dow_labels': list(_SELECTOR_DOW_LABELS),
    }


def _cluster_38h_profiles_no_plots(
    df_wide: pd.DataFrame,
    match_dates_set: set,
    df_holidays_local: pd.DataFrame,
    hour_cols: list,
    n_clusters: int,
    previously_w_hours: int,
) -> dict:
    cluster_labels = list('CDEFGHIJ')
    eve_hour_cols = hour_cols[-previously_w_hours:]
    hol_hour_cols = hour_cols
    feat_cols = [f'eve_{col_name}' for col_name in eve_hour_cols] + [f'hol_{col_name}' for col_name in hol_hour_cols]

    date_to_name = dict(
        zip(
            pd.to_datetime(df_holidays_local['date']).dt.normalize(),
            df_holidays_local['holiday_name'],
        )
    )
    wide_idx = set(df_wide.index.normalize())
    rows = []
    for target_date in sorted(match_dates_set):
        eve_date = target_date - pd.Timedelta(days=1)
        if eve_date not in wide_idx:
            continue
        row_eve = df_wide.loc[df_wide.index.normalize() == eve_date].squeeze()
        row_hol = df_wide.loc[df_wide.index.normalize() == target_date].squeeze()
        eve_values = row_eve[eve_hour_cols].values.astype(float)
        holiday_values = row_hol[hol_hour_cols].values.astype(float)
        if np.isnan(eve_values).any() or np.isnan(holiday_values).any():
            continue
        rows.append({
            'date': target_date,
            'holiday_name': date_to_name.get(target_date, str(target_date.date())),
            **dict(zip([f'eve_{col_name}' for col_name in eve_hour_cols], eve_values)),
            **dict(zip([f'hol_{col_name}' for col_name in hol_hour_cols], holiday_values)),
        })

    if not rows:
        df_phol = pd.DataFrame(columns=['holiday_name', *feat_cols, 'cluster', 'cluster_type'])
        df_phol.index = pd.DatetimeIndex([], name='date')
        return {
            'df_phol': df_phol,
            'df_days_phol': pd.DataFrame(columns=['type', 'date', 'dow', 'holiday_name']),
            'kmeans_phol': None,
            'centroids_phol': np.empty((0, len(feat_cols))),
            'feat_cols': feat_cols,
        }

    df_phol = pd.DataFrame(rows).set_index('date')
    resolved_n_clusters = min(int(n_clusters), len(df_phol))
    profiles = df_phol[feat_cols].values.astype(float)
    profiles_scaled = identify_holidays_module.StandardScaler().fit_transform(profiles)
    kmeans_phol = identify_holidays_module.KMeans(
        n_clusters=resolved_n_clusters,
        random_state=42,
        n_init=20,
    ).fit(profiles_scaled)
    df_phol['cluster'] = kmeans_phol.labels_
    df_phol['cluster_type'] = [cluster_labels[label] for label in kmeans_phol.labels_]

    centroids_phol = np.array([
        df_phol.loc[df_phol['cluster'] == cluster_id, feat_cols].values.mean(axis=0)
        for cluster_id in range(resolved_n_clusters)
    ])

    df_days_phol = (
        pd.DataFrame(
            [
                {
                    'type': row['cluster_type'],
                    'date': date_value,
                    'dow': date_value.day_name()[:3],
                    'holiday_name': row['holiday_name'],
                }
                for date_value, row in df_phol.iterrows()
            ]
        )
        .sort_values(['type', 'date'])
        .reset_index(drop=True)
    )

    return {
        'df_phol': df_phol,
        'df_days_phol': df_days_phol,
        'kmeans_phol': kmeans_phol,
        'centroids_phol': centroids_phol,
        'feat_cols': feat_cols,
    }


def _build_selector_history_for_unique_id(unique_id: str) -> dict:
    df_raw_full = load_results_data(DEMAND_PATH, unique_id)
    available_dates_full = set(df_raw_full['ds'].dt.normalize())

    df_raw_local = df_raw_full.copy()
    if DATE_END is not None:
        cutoff_ts = pd.Timestamp(DATE_END)
        df_raw_local = df_raw_local[df_raw_local['ds'] < cutoff_ts].copy()
    if df_raw_local.empty:
        raise ValueError(f'No data available for {unique_id} inside the historical selector window.')

    df_wide_local = build_wide_df(df_raw_local, MONTH_TO_SEGMENT, exclude_years=[])
    if df_wide_local.empty:
        raise ValueError(f'No complete daily profiles are available for {unique_id}.')

    hour_cols_local = get_hour_cols(df_wide_local)
    year_min_local = int(df_wide_local.index.year.min())
    year_max_local = int(df_wide_local.index.year.max())
    df_holidays_local = load_holidays_catalog(HOLIDAYS_PATH, year_min_local, year_max_local)
    known_holiday_dates_local = (
        set(pd.to_datetime(df_holidays_local['date']).dt.normalize())
        if EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE else set()
    )

    df_dist_local = compute_distances(
        df_wide_local,
        hour_cols_local,
        SEGMENT_LABELS,
        WEEKDAY_NAMES,
        distance_metric=DISTANCE_METRIC,
        reference_exclude_dates=known_holiday_dates_local,
    )
    df_dist_local, df_outliers_local = detect_outliers(
        df_dist_local,
        OUTLIER_PERCENTILE,
        threshold_reference_only=EXCLUDE_KNOWN_HOLIDAYS_FROM_BASELINE,
        promote_dates=known_holiday_dates_local,
        promote_min_group_percentile=KNOWN_HOLIDAY_MIN_GROUP_PERCENTILE,
    )
    df_match_local, _ = compare_outliers_holidays(df_outliers_local, df_holidays_local)
    all_dates_local = set(df_wide_local.index.normalize())
    date_sets_local = get_date_sets(df_match_local, df_holidays_local, all_dates_local)

    match_dates_set_local = date_sets_local['match_dates_set']
    unknown_dates_set_local = date_sets_local['unknown_dates_set']
    outlier_dates_set_local = date_sets_local['outlier_dates_set']
    atypical_dates_local = match_dates_set_local | unknown_dates_set_local

    if atypical_dates_local:
        cluster_results_local = identify_holidays_module.cluster_atypical_profiles(
            df_wide_local,
            match_dates_set_local,
            unknown_dates_set_local,
            outlier_dates_set_local,
            min(selector_n_clusters, len(atypical_dates_local)),
            hour_cols_local,
            df_holidays_local,
        )
        cluster_ab_results_local = identify_holidays_module.classify_cluster_dow_type(
            df_wide_local,
            cluster_results_local['df_atyp'],
            outlier_dates_set_local,
            hour_cols_local,
            selector_ab_alpha,
        )
    else:
        cluster_ab_results_local = _empty_cluster_ab_results()

    cluster_38h_results_local = _cluster_38h_profiles_no_plots(
        df_wide_local,
        match_dates_set_local,
        df_holidays_local,
        hour_cols_local,
        N_CLUSTERS_PHOL,
        PREVIOUSLY_W_HOURS,
    )

    df_selector_history_local = build_holiday_selector_features(
        df_wide=df_wide_local,
        df_holidays=df_holidays_local,
        cluster_ab_results=cluster_ab_results_local,
        df_phol=cluster_38h_results_local['df_phol'],
        holidays_path=HOLIDAYS_PATH,
        unique_id=unique_id,
    )

    print(
        f'[{unique_id}] complete days={len(df_wide_local)} | '
        f'history selector rows={len(df_selector_history_local)} | '
        f'matched holidays={len(match_dates_set_local)}'
    )
    return {
        'unique_id': unique_id,
        'selector_history': df_selector_history_local,
        'available_dates_full': available_dates_full,
    }


selector_series_contexts = {}
selector_history_frames = []
for series_unique_id in AVAILABLE_UNIQUE_IDS:
    selector_context = _build_selector_history_for_unique_id(series_unique_id)
    selector_series_contexts[series_unique_id] = selector_context
    selector_history_frames.append(selector_context['selector_history'])

df_holiday_selector_features_history_all = (
    pd.concat(selector_history_frames, ignore_index=True)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)
df_holiday_selector_features_history = (
    df_holiday_selector_features_history_all
    .loc[df_holiday_selector_features_history_all['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

selector_history_counts = (
    df_holiday_selector_features_history_all
    .groupby('unique_id', dropna=False)
    .size()
    .rename('historical_rows')
    .reset_index()
    .sort_values('unique_id')
    .reset_index(drop=True)
)

print(f'Historical selector rows ({UNIQUE_ID}): {len(df_holiday_selector_features_history)}')
print(
    f'Historical selector rows (all series): {len(df_holiday_selector_features_history_all)} '
    f'| series={df_holiday_selector_features_history_all["unique_id"].nunique()}'
)
display(selector_history_counts)
display(df_holiday_selector_features_history)

[ERCOT_demand_COAST] complete days=3366 | history selector rows=156 | matched holidays=56
[ERCOT_demand_EAST] complete days=3366 | history selector rows=156 | matched holidays=61
[ERCOT_demand_FWEST] complete days=3366 | history selector rows=156 | matched holidays=57
[ERCOT_demand_NORTH] complete days=3366 | history selector rows=156 | matched holidays=51
[ERCOT_demand_NCENT] complete days=3366 | history selector rows=156 | matched holidays=61
[ERCOT_demand_SOUTH] complete days=3366 | history selector rows=156 | matched holidays=51
[ERCOT_demand_SCENT] complete days=3366 | history selector rows=156 | matched holidays=62
[ERCOT_demand_WEST] complete days=3366 | history selector rows=156 | matched holidays=51
[ERCOT_demand_ERCOT] complete days=3366 | history selector rows=156 | matched holidays=66
Historical selector rows (ERCOT_demand_ERCOT): 156
Historical selector rows (all series): 1404 | series=9


,unique_id,historical_rows
0,ERCOT_demand_COAST,156
1,ERCOT_demand_EAST,156
2,ERCOT_demand_ERCOT,156
3,ERCOT_demand_FWEST,156
4,ERCOT_demand_NCENT,156
5,ERCOT_demand_NORTH,156
6,ERCOT_demand_SCENT,156
7,ERCOT_demand_SOUTH,156
8,ERCOT_demand_WEST,156


,unique_id,holiday_name,anchor_holiday_name,date,holiday_day_type,weekday_name,day_class_code,day_class_name,season,date_rule,is_fixed_date,is_observed_monday_rule,best_matching_weekday,daily_profile_cluster,daily_profile_cluster_id,daily_profile_archetype,event_profile_cluster,event_profile_cluster_id
0,ERCOT_demand_ERCOT,Martin Luther King Jr. Day,Martin Luther King Jr. Day,2016-01-18,H2,Monday,1,Weekday,Winter,relative_weekday_rule,False,False,NaN,NaN,<NA>,NaN,NaN,<NA>
1,ERCOT_demand_ERCOT,Presidents' Day,Presidents' Day,2016-02-15,H2,Monday,1,Weekday,Winter,relative_weekday_rule,False,False,NaN,NaN,<NA>,NaN,NaN,<NA>
2,ERCOT_demand_ERCOT,Texas Independence Day,Texas Independence Day,2016-03-02,H2,Wednesday,1,Weekday,Spring,fixed_date,True,False,NaN,NaN,<NA>,NaN,NaN,<NA>
3,ERCOT_demand_ERCOT,San Jacinto Day,San Jacinto Day,2016-04-21,H2,Thursday,1,Weekday,Spring,fixed_date,True,False,NaN,NaN,<NA>,NaN,NaN,<NA>
4,ERCOT_demand_ERCOT,Memorial Day,Memorial Day,2016-05-30,H2,Monday,1,Weekday,Spring,relative_weekday_rule,False,False,Sunday,B,1,unclear,E,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
151,ERCOT_demand_ERCOT,Post-holiday recovery,Christmas Day,2025-12-26,H4,Friday,1,Weekday,Winter,derived_recovery_day,False,False,NaN,NaN,<NA>,NaN,NaN,<NA>
152,ERCOT_demand_ERCOT,New Year's Day,New Year's Day,2026-01-01,H2,Thursday,1,Weekday,Winter,fixed_date,True,False,NaN,NaN,<NA>,NaN,NaN,<NA>
153,ERCOT_demand_ERCOT,Martin Luther King Jr. Day,Martin Luther King Jr. Day,2026-01-19,H2,Monday,1,Weekday,Winter,relative_weekday_rule,False,False,NaN,NaN,<NA>,NaN,NaN,<NA>
154,ERCOT_demand_ERCOT,Presidents' Day,Presidents' Day,2026-02-16,H2,Monday,1,Weekday,Winter,relative_weekday_rule,False,False,NaN,NaN,<NA>,NaN,NaN,<NA>


In [18]:
# 10. Ex-ante profile priors for new candidates
# Priors are estimated on the historical selector rows of all available series.
from analog_holidays.shared.identify_holidays import (
    build_future_holiday_selector_features,
    build_holiday_selector_priors,
)

selector_features_path = Path('holidays') / 'holiday_selector_features_ercot.csv'
selector_priors_path = Path('holidays') / 'holiday_selector_priors_ercot.csv'

if 'selector_series_contexts' not in globals() or not selector_series_contexts:
    raise ValueError('Run Cell 33 first to build the per-series historical selector contexts.')

df_holiday_selector_priors = build_holiday_selector_priors(
    df_holiday_selector_features_history_all,
    group_cols=_SELECTOR_GROUP_COLS,
)

selector_future_start = (
    pd.Timestamp(DATE_END).normalize()
    if DATE_END is not None
    else pd.to_datetime(df_holiday_selector_features_history_all['date']).max().normalize() + pd.Timedelta(days=1)
)
selector_future_end = pd.Timestamp(SELECTOR_FUTURE_END).normalize() if SELECTOR_FUTURE_END is not None else None
future_year_end = (
    int(selector_future_end.year)
    if selector_future_end is not None
    else int(pd.to_datetime(df_holiday_selector_features_history_all['date']).max().year)
)
df_holidays_selector_source = load_holidays_catalog(
    HOLIDAYS_PATH,
    year_min,
    future_year_end,
)

future_frames = []
for series_unique_id, series_context in selector_series_contexts.items():
    df_future_local = build_future_holiday_selector_features(
        df_holidays=df_holidays_selector_source,
        df_priors=df_holiday_selector_priors,
        available_dates=series_context['available_dates_full'],
        holidays_path=HOLIDAYS_PATH,
        group_cols=_SELECTOR_GROUP_COLS,
        start_date=selector_future_start,
        end_date=selector_future_end,
        unique_id=series_unique_id,
    )
    future_frames.append(df_future_local)
    print(f'[{series_unique_id}] future ex-ante rows={len(df_future_local)}')

df_holiday_selector_features_future_all = (
    pd.concat(future_frames, ignore_index=True)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
    if future_frames else pd.DataFrame(columns=df_holiday_selector_features_history_all.columns)
)
df_holiday_selector_features_future = (
    df_holiday_selector_features_future_all
    .loc[df_holiday_selector_features_future_all['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

df_holiday_selector_features = (
    pd.concat(
        [df_holiday_selector_features_history_all, df_holiday_selector_features_future_all],
        ignore_index=True,
    )
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)

df_holiday_selector_features_export = df_holiday_selector_features.copy()
df_holiday_selector_features_export['date'] = (
    pd.to_datetime(df_holiday_selector_features_export['date'])
    .dt.strftime('%Y-%m-%d')
)
df_holiday_selector_features_export.to_csv(selector_features_path, index=False)

df_holiday_selector_priors_export = df_holiday_selector_priors.copy()
df_holiday_selector_priors_export.to_csv(selector_priors_path, index=False)

selector_future_counts = (
    df_holiday_selector_features_future_all
    .groupby('unique_id', dropna=False)
    .size()
    .rename('future_rows')
    .reset_index()
    .sort_values('unique_id')
    .reset_index(drop=True)
    if not df_holiday_selector_features_future_all.empty else pd.DataFrame(columns=['unique_id', 'future_rows'])
)

print(f'Historical selector rows ({UNIQUE_ID}): {len(df_holiday_selector_features_history)}')
print(f'Future ex-ante rows ({UNIQUE_ID}): {len(df_holiday_selector_features_future)}')
print(f'Total selector rows exported: {len(df_holiday_selector_features_export)}')
print(
    f'Series exported: {df_holiday_selector_features_export["unique_id"].nunique()} '
    f'| selector_future_start={selector_future_start.date()} '
    f'| selector_future_end={selector_future_end.date() if selector_future_end is not None else "source max"}'
)
print(f'Selector CSV saved to: {selector_features_path.resolve()}')
print(f'Priors CSV saved to: {selector_priors_path.resolve()}')
display(selector_future_counts)
display(
    df_holiday_selector_priors
    .loc[df_holiday_selector_priors['unique_id'] == UNIQUE_ID]
    .reset_index(drop=True)
)
display(df_holiday_selector_features_future)

[ERCOT_demand_COAST] future ex-ante rows=0
[ERCOT_demand_EAST] future ex-ante rows=0
[ERCOT_demand_FWEST] future ex-ante rows=0
[ERCOT_demand_NORTH] future ex-ante rows=0
[ERCOT_demand_NCENT] future ex-ante rows=0
[ERCOT_demand_SOUTH] future ex-ante rows=0
[ERCOT_demand_SCENT] future ex-ante rows=0
[ERCOT_demand_WEST] future ex-ante rows=0
[ERCOT_demand_ERCOT] future ex-ante rows=0
Historical selector rows (ERCOT_demand_ERCOT): 156
Future ex-ante rows (ERCOT_demand_ERCOT): 0
Total selector rows exported: 1404
Series exported: 9 | selector_future_start=2026-03-03 | selector_future_end=2026-12-31
Selector CSV saved to: /home/uriel/GIT/analog_holidays/holidays/holiday_selector_features_ercot.csv
Priors CSV saved to: /home/uriel/GIT/analog_holidays/holidays/holiday_selector_priors_ercot.csv


,unique_id,future_rows


,unique_id,anchor_holiday_name,holiday_day_type,history_rows,history_years,inferred_best_matching_weekday,inferred_daily_profile_cluster,inferred_daily_profile_cluster_id,inferred_daily_profile_archetype,inferred_event_profile_cluster,inferred_event_profile_cluster_id
0,ERCOT_demand_ERCOT,Christmas Day,H2,9,9,Tuesday,A,0,unclear,D,1
1,ERCOT_demand_ERCOT,Christmas Day,H4,9,9,Tuesday,A,0,unclear,D,1
2,ERCOT_demand_ERCOT,Christmas Eve,H1,9,9,Tuesday,A,0,unclear,E,2
3,ERCOT_demand_ERCOT,Day after Thanksgiving,H2,9,9,Wednesday,A,0,unclear,D,1
4,ERCOT_demand_ERCOT,Day after Thanksgiving,H4,9,9,Wednesday,A,0,unclear,D,1
5,ERCOT_demand_ERCOT,Independence Day,H2,9,9,Saturday,B,1,unclear,C,0
6,ERCOT_demand_ERCOT,Juneteenth National Independence Day,H2,9,9,Friday,B,1,unclear,D,1
7,ERCOT_demand_ERCOT,Labor Day,H2,9,9,Saturday,B,1,unclear,D,1
8,ERCOT_demand_ERCOT,Lyndon B. Johnson Day,H2,9,9,Friday,B,1,unclear,C,0
9,ERCOT_demand_ERCOT,Martin Luther King Jr. Day,H2,10,10,Friday,A,0,unclear,E,2


,unique_id,holiday_name,anchor_holiday_name,date,holiday_day_type,weekday_name,day_class_code,day_class_name,season,date_rule,is_fixed_date,is_observed_monday_rule,best_matching_weekday,daily_profile_cluster,daily_profile_cluster_id,daily_profile_archetype,event_profile_cluster,event_profile_cluster_id


## 11. Analog-Space Clusters (`F`, `G`, `H`, ...)

Use `CLUSTERING_CRITERIUM` to choose how daily selector rows are grouped into stable analog-space labels.

The public criterion catalog exposed in this notebook currently includes `shape_pearson_CDE_map_FGH`, `seasonal_heat_cold`, `seasonal_winter_sprint_fall`, and `best_matching_weekday`. The binary seasonal option collapses Spring/Summer into heat and Autumn/Winter into cold.

Historical rows determine the analog-space mapping. Future holiday rows in the 2025/2026 test horizon receive their profile labels ex-ante from `holiday_selector_priors.csv`, so the exported selector can be filled without using future observations.

The selector export `holidays/holiday_selector_features.csv` is the single source of truth for daily `analog_cluster` labels. No hourly `*_cluster` columns are added to the demand source anymore.

In [19]:
from analog_holidays.shared.identify_holidays import assign_holiday_selector_analog_clusters

analog_cluster_results = assign_holiday_selector_analog_clusters(
    df_selector=df_holiday_selector_features,
    df_priors=df_holiday_selector_priors,
    criterion=CLUSTERING_CRITERIUM,
    group_cols=_SELECTOR_GROUP_COLS,
    cluster_labels=('F', 'G', 'H'),
)
df_holiday_selector_analog_clusters = analog_cluster_results['df_selector_clusters']
df_analog_cluster_catalog = analog_cluster_results['analog_cluster_catalog']
analog_criterion_prior_col = analog_cluster_results.get('analog_criterion_prior_col')

drop_columns = [
    'analog_criterion',
    'analog_criterion_value',
]
if analog_criterion_prior_col is not None:
    drop_columns.append(analog_criterion_prior_col)

df_holiday_selector_features = (
    df_holiday_selector_analog_clusters
    .drop(columns=drop_columns, errors='ignore')
    .assign(analog_cluster_criterion=CLUSTERING_CRITERIUM)
    .sort_values(['unique_id', 'date', 'holiday_name'])
    .reset_index(drop=True)
)
df_holiday_selector_features_current = (
    df_holiday_selector_features
    .loc[df_holiday_selector_features['unique_id'] == UNIQUE_ID]
    .copy()
    .reset_index(drop=True)
)

_analog_cluster_by_date = (
    df_holiday_selector_features_current[['date', 'analog_cluster']]
    .dropna(subset=['analog_cluster'])
    .drop_duplicates(subset=['date'])
    .copy()
)
_analog_cluster_by_date['date'] = pd.to_datetime(_analog_cluster_by_date['date']).dt.normalize()

df_holidays_display = (
    df_holidays_display
    .drop(columns=['analog_cluster'], errors='ignore')
    .merge(_analog_cluster_by_date, on='date', how='left')
)

df_holiday_selector_features_export = df_holiday_selector_features.copy()
if 'date' in df_holiday_selector_features_export.columns:
    df_holiday_selector_features_export['date'] = (
        pd.to_datetime(df_holiday_selector_features_export['date'])
        .dt.strftime('%Y-%m-%d')
    )
df_holiday_selector_features_export.to_csv(selector_features_path, index=False)

df_analog_cluster_summary = (
    df_holiday_selector_features
    .dropna(subset=['analog_cluster'])
    .groupby(['unique_id', 'analog_cluster'], dropna=False)
    .size()
    .rename('n_rows')
    .reset_index()
    .sort_values(['unique_id', 'analog_cluster'])
    .reset_index(drop=True)
)

print(f'Analog-cluster criterion: {CLUSTERING_CRITERIUM}')
print(f'Catalog rows: {len(df_analog_cluster_catalog)}')
print(
    f'Analog labels exported: {tuple(df_analog_cluster_catalog["analog_cluster"].astype(str).tolist())}'
)
print(
    f'Selector CSV saved to: {selector_features_path.resolve()} '
    f'| series exported={df_holiday_selector_features_export["unique_id"].nunique()}'
)
print('df_holidays_display updated with analog_cluster for the current detail-view series. Re-run the figure cells in sections 7 and 8 to see labels like [F].')
display(df_analog_cluster_catalog)
display(df_analog_cluster_summary)
display(
    df_holiday_selector_features_current[
        [
            'unique_id',
            'holiday_name',
            'anchor_holiday_name',
            'date',
            'holiday_day_type',
            'event_profile_cluster',
            'analog_cluster_criterion',
            'analog_cluster',
        ]
    ].sort_values('date').reset_index(drop=True)
)

Analog-cluster criterion: best_matching_weekday
Catalog rows: 7
Analog labels exported: ('F', 'G', 'H', 'I', 'J', 'K', 'L')
Selector CSV saved to: /home/uriel/GIT/analog_holidays/holidays/holiday_selector_features_ercot.csv | series exported=9
df_holidays_display updated with analog_cluster for the current detail-view series. Re-run the figure cells in sections 7 and 8 to see labels like [F].


,analog_cluster,analog_criterion,analog_criterion_value,n_rows,n_anchor_holidays
0,F,best_matching_weekday,Friday,263,15
1,G,best_matching_weekday,Monday,144,10
2,H,best_matching_weekday,Saturday,254,13
3,I,best_matching_weekday,Sunday,311,15
4,J,best_matching_weekday,Thursday,75,11
5,K,best_matching_weekday,Tuesday,161,10
6,L,best_matching_weekday,Wednesday,187,11


,unique_id,analog_cluster,n_rows
0,ERCOT_demand_COAST,F,19
1,ERCOT_demand_COAST,G,41
2,ERCOT_demand_COAST,H,6
3,ERCOT_demand_COAST,I,49
4,ERCOT_demand_COAST,J,2
...,...,...,...
56,ERCOT_demand_WEST,H,39
57,ERCOT_demand_WEST,I,24
58,ERCOT_demand_WEST,J,14
59,ERCOT_demand_WEST,K,12


,unique_id,holiday_name,anchor_holiday_name,date,holiday_day_type,event_profile_cluster,analog_cluster_criterion,analog_cluster
0,ERCOT_demand_ERCOT,Martin Luther King Jr. Day,Martin Luther King Jr. Day,2016-01-18,H2,NaN,best_matching_weekday,F
1,ERCOT_demand_ERCOT,Presidents' Day,Presidents' Day,2016-02-15,H2,NaN,best_matching_weekday,F
2,ERCOT_demand_ERCOT,Texas Independence Day,Texas Independence Day,2016-03-02,H2,NaN,best_matching_weekday,L
3,ERCOT_demand_ERCOT,San Jacinto Day,San Jacinto Day,2016-04-21,H2,NaN,best_matching_weekday,H
4,ERCOT_demand_ERCOT,Memorial Day,Memorial Day,2016-05-30,H2,E,best_matching_weekday,I
...,...,...,...,...,...,...,...,...
151,ERCOT_demand_ERCOT,Post-holiday recovery,Christmas Day,2025-12-26,H4,NaN,best_matching_weekday,K
152,ERCOT_demand_ERCOT,New Year's Day,New Year's Day,2026-01-01,H2,NaN,best_matching_weekday,I
153,ERCOT_demand_ERCOT,Martin Luther King Jr. Day,Martin Luther King Jr. Day,2026-01-19,H2,NaN,best_matching_weekday,F
154,ERCOT_demand_ERCOT,Presidents' Day,Presidents' Day,2026-02-16,H2,NaN,best_matching_weekday,F


In [20]:
analog_cluster_labels = tuple(
    df_analog_cluster_catalog['analog_cluster']
    .dropna()
    .astype(str)
    .tolist()
)
if not analog_cluster_labels:
    analog_cluster_labels = ('F', 'G', 'H')

analog_cluster_fgh_results = run_analog_cluster_38h_analysis(
    df_phol=df_phol,
    df_holiday_selector_features=df_holiday_selector_features,
    cluster_colors=CLUSTER_COLORS,
    unique_id=UNIQUE_ID,
    feat_cols=cluster_38h_results['feat_cols'],
    previously_w_hours=PREVIOUSLY_W_HOURS,
    cluster_labels=analog_cluster_labels,
    selection_criterion=CLUSTERING_CRITERIUM,
)

df_phol_fgh = analog_cluster_fgh_results['df_phol_fgh']
df_days_fgh = analog_cluster_fgh_results['df_days_fgh']
centroids_fgh = analog_cluster_fgh_results['centroids_fgh']
feat_cols = analog_cluster_fgh_results['feat_cols']


Days per analog cluster


type,date,dow,holiday_name,cluster_38h
F,2017-11-23,Thu,Thanksgiving Day,E
F,2018-02-19,Mon,Presidents' Day,E
F,2018-11-22,Thu,Thanksgiving Day,E
F,2019-08-27,Tue,Lyndon B. Johnson Day,C
F,2022-11-24,Thu,Thanksgiving Day,E
F,2023-11-23,Thu,Thanksgiving Day,D
F,2024-02-19,Mon,Presidents' Day,D
F,2025-02-17,Mon,Presidents' Day,D
F,2025-11-27,Thu,Thanksgiving Day,D
G,2018-06-19,Tue,Juneteenth National Independence Day,D


Analog-cluster profiles available: 65


/home/uriel/GIT/analog_holidays/shared/identify_holidays.py:2415: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show()


## 12. Final summary

In [21]:
print_summary(
    UNIQUE_ID, df_wide, n_total, n_match, n_unknown,
    df_missed, df_outliers_cmp, OUTLIER_PERCENTILE, WEEKDAY_NAMES,
)

Holiday identification summary
Series: ERCOT_demand_ERCOT
Range: 2016-01-02 to 2026-03-31
Days analyzed: 3366
Outlier threshold: percentile 95
Detected outliers: 245
Match the holiday catalog: 66
New candidates: 179
Known holidays not detected: 72

Outlier dates
  2016-03-21 (Monday)
  2016-04-01 (Friday)
  2016-04-02 (Saturday)
  2016-05-02 (Monday)
  2016-05-14 (Saturday)
  2016-05-19 (Thursday)
  2016-05-30 (Monday)
  2016-05-31 (Tuesday)
  2016-06-02 (Thursday)
  2016-06-03 (Friday)
  2016-06-28 (Tuesday)
  2016-07-04 (Monday)
  2016-07-25 (Monday)
  2016-08-15 (Monday)
  2016-08-16 (Tuesday)
  2016-08-17 (Wednesday)
  2016-08-18 (Thursday)
  2016-08-19 (Friday)
  2016-08-20 (Saturday)
  2016-08-22 (Monday)
  2016-09-25 (Sunday)
  2016-12-20 (Tuesday)
  2017-01-01 (Sunday)
  2017-01-08 (Sunday)
  2017-01-16 (Monday)
  2017-03-02 (Thursday)
  2017-05-03 (Wednesday)
  2017-05-29 (Monday)
  2017-06-02 (Friday)
  2017-06-24 (Saturday)
  2017-07-04 (Tuesday)
  2017-07-15 (Saturday)
  20